# Visualize Process Graphs

In Microsoft Defender XDR logs, you can see the ID of the process as well as the ID of its parent.

Using these values, you can visualize a "spawn" graph, showing which processes are spawning other processes.

### Optional: Use AlienVault OTX indicators of compromise for files

Optionally, you can choose to load file MD5 hash indicators of compromise from the AlienVault OTX API.

You will need an AlienVault OTX API key (easy and free to do).

If you want to do this, set `LOAD_FILE_MD5_INDICATORS_FROM_ALIEN_VAULT_OTX` to `true`, and add your `ALIEN_VAULT_OTX_API_KEY` below:

In [ ]:
LOAD_FILE_MD5_INDICATORS_FROM_ALIEN_VAULT_OTX = False
ALIEN_VAULT_OTX_API_KEY = None # os.environ.get("ALIEN_VAULT_OTX_API_KEY")

### Install libraries

In [ ]:
%pip install scanner-client yfiles_jupyter_graphs pandas OTXv2

In [ ]:
import os
import pandas as pd
from scanner_client import Scanner
from yfiles_jupyter_graphs import GraphWidget
from datetime import datetime, timezone, timedelta

Add utility function to convert Scanner search results to a pandas data frame.

In [ ]:
def convert_results_to_data_frame(results):
    rows = [row.columns.to_dict() for row in results.rows]
    column_tags = results.column_tags.to_dict()
    if len(column_tags) > 0:
        # If this is a table, use the column ordering in the data frame
        return pd.DataFrame(data=rows, columns=results.column_ordering)
    else:
        # Otherwise, this is a list of log events, so use pandas JSON
        # normalization to set the table columns to the union of all keys.
        return pd.json_normalize(rows)


Initialize Scanner API client:

In [ ]:
scanner = Scanner(
    api_url=os.environ["SCANNER_API_URL"],
    api_key=os.environ["SCANNER_API_KEY"],
)

Set analyzed time range to be the last 7 days.

In [ ]:
end_time = datetime.now(tz=timezone.utc)
start_time = end_time - timedelta(days=7)

Find the device with the largest amount of activity.

In [ ]:
response = scanner.query.blocking_query(
    start_time=start_time.isoformat(),
    end_time=end_time.isoformat(),
    query_text="""
        %ingest.source_type: "mde"
        DeviceName=*
        | stats
          min(timestamp) as firstTime,
          max(timestamp) as lastTime,
          count() as activityCount
          by
          DeviceId
    """
)
device_activity_df = convert_results_to_data_frame(response.results)

In [ ]:
device_activity_df.head()

In [ ]:
device_id = device_activity_df["DeviceId"][0]
device_id

Query for logs for that device containing information about process IDs and parent IDs.

In [ ]:
response = scanner.query.blocking_query(
    start_time=start_time.isoformat(),
    end_time=end_time.isoformat(),
    query_text=f"""
        %ingest.source_type: "mde"
        DeviceId="{device_id}"
        InitiatingProcessId=*
        InitiatingProcessParentId=*
        | rename 
          InitiatingProcessId as processId,
          InitiatingProcessParentId as parentProcessId,
          InitiatingProcessFolderPath as processPath,
          InitiatingProcessParentFolderPath as parentProcessPath,
          InitiatingProcessMD5 as processMd5
        | stats
          min(timestamp) as firstTime,
          max(timestamp) as lastTime
          by
          processId,
          parentProcessId,
          processPath,
          parentProcessPath,
          processMd5
    """
)
process_df = convert_results_to_data_frame(response.results)

In [ ]:
process_df.head()

In [ ]:
len(process_df)

### Optional: Collect File Hash MD5 Indicators of Compromise

Optionally, load file MD5 hash indicators of compromise from AlienVault OTX.

You will need an AlienVault OTX API key (easy and free to do).

Specifically, we will load file MD5 hash indicators published by AlienVault since the last 30 days.

In [ ]:
from OTXv2 import OTXv2, IndicatorTypes
from pprint import pprint

file_md5_indicators = set()
otx_client = None
if LOAD_FILE_MD5_INDICATORS_FROM_ALIEN_VAULT_OTX:
    otx_client = OTXv2(ALIEN_VAULT_OTX_API_KEY)
    now = datetime.now()
    modified_since = datetime.now() - timedelta(days=30)
    file_hash_info_iter = otx_client.get_all_indicators(
        author_name="AlienVault", 
        indicator_types=[IndicatorTypes.FILE_HASH_MD5], 
        modified_since=modified_since
    )
    count = 0
    for file_hash_info in file_hash_info_iter:
        if file_hash_info.get('is_active') == 1:
            file_md5_indicators.add(file_hash_info['indicator'])
    print(len(file_md5_indicators))

def get_file_md5_indicator_summary(file_md5):
    if not otx_client:
        return {}
    result = otx_client.get_indicator_details_by_section(
        IndicatorTypes.FILE_HASH_MD5, 
        file_md5,
        "general"
    )
    pulses = result.get('pulse_info', {}).get('pulses', [])
    first_pulse = pulses[0] if pulses else {}
    name = first_pulse.get('name', '')
    description = first_pulse.get('description', '')
    references = first_pulse.get('references', [])
    return {
        'name': name,
        'description': description,
        'references': references,
    }

Compute nodes and edges of process spawn graph.

In [ ]:
from collections import defaultdict

temp_dir_examples = [
    "\\systemtemp\\", 
    "\\temp\\", 
    "\\tmp\\"
]

def compute_risk_score(process_path, process_md5):
    if process_md5 in file_md5_indicators:
        return 10
    elif any(temp_dir in process_path for temp_dir in temp_dir_examples):
        return 5
    else:
        return 0

node_counts = defaultdict(int)
edges = []
printed = False
for i, row in enumerate(process_df.to_dict(orient="records")):
    if not row['processId'] or not row['parentProcessId']:
        next
    node_counts[row['processId']] += 1
    node_counts[row['parentProcessId']] += 1
    if not printed:
        printed = True
        print(row['processPath'])
    edges.append({
        'id': i,
        'start': row['parentProcessId'],
        'end': row['processId'],
        'properties': {
            'process_id': row['processId'],
            'process_md5': row['processMd5'],
            'parent_process_id': row['parentProcessId'],
            'process_path': row['processPath'],
            'parent_process_path': row['parentProcessPath'],
        }
    })

node_ids = set(process_df['processId'].unique()).union(set(process_df['parentProcessId'].unique()))
nodes = []
file_md5_indicator_summaries = {}
for node_id in node_ids:
    if not node_id:
        continue
    process_path = process_df[process_df['processId'] == node_id]['processPath'].values[0] if node_id in process_df['processId'].values else ''
    process_md5 = process_df[process_df['processId'] == node_id]['processMd5'].values[0] if node_id in process_df['processId'].values else ''

    risk_score = compute_risk_score(process_path, process_md5)
    if process_md5 in file_md5_indicators and process_md5 not in file_md5_indicator_summaries:
        file_md5_indicator_summaries[process_md5] = get_file_md5_indicator_summary(process_md5)
        
    node = {
        'id': node_id,
        'properties': {
            'process_id': node_id, 
            'process_path': process_path,
            'process_md5': process_md5,
            'risk_score': risk_score,
            'threat_summary': file_md5_indicator_summaries.get(process_md5),
            'label': f"{process_path} ({node_id})",
            'count': node_counts[node_id],
        },
    }
    nodes.append(node)

min_count = min(node_counts.values())
max_count = max(node_counts.values()) 

min_scale_factor = 1.0
max_scale_factor = 10.0

def scale_factor_mapping(node):
    if node['properties']['risk_score'] >= 10:
        return max_scale_factor
    elif node['properties']['risk_score'] >= 5:
        return (max_scale_factor - min_scale_factor) / 2.0
    count = node['properties']['count']
    numer = count - min_count
    denom = max_count - min_count
    if denom == 0:
        return 1.0  
    frac = numer / denom
    delta = (max_scale_factor - min_scale_factor) * frac
    return min_scale_factor + delta

def styles_mapping(node):
    risk_score = node.get('properties', {}).get('risk_score', 0)
    yellow = '#ffff00'
    red = '#ff0000'
    if risk_score >= 10:
        return { 'color': red }
    elif risk_score >= 5:
        return { 'color': yellow }
    else:
        return {}
    

Generate interactive graph visualization of process spawn graph.

### Node Colors

1. RED: Processes with MD5 hashes that appear in our indicators of compromise set.
2. YELLOW: Processes running in temporary directories.

### Interaction

- Click and drag to navigate. Use mouse wheel to zoom in/out.
- Click on a node or an edge to select it and see what it is connected to.
- When a node or edge is selected, inspect its properties in the Data tab in the side bar.
- Search for a node or edge via the Search tab in the side bar.
- Change the layout of the graph to examine relationships in different ways.

In [ ]:
w = GraphWidget()
w.nodes = nodes
w.edges = edges
w.directed = True
w.set_node_scale_factor_mapping(scale_factor_mapping)
w.set_node_styles_mapping(styles_mapping)
w